20240812：  
用于采集brenda和sabio-rk数据库的kcat数据的脚本

### BRENDA采集数据不要用这个脚本，用2.7版本的python文件@240820

### get EC numbers and fetch data from Brenda

In [1]:
############################################################################################################
# get EC numbers from Brenda
# fetch data from brenda(including kcat and km)
output_file = 'kcat_km_values.csv'

import hashlib
import pandas as pd
from zeep import Client
from zeep.exceptions import Fault, TransportError
from requests.exceptions import ConnectionError, ChunkedEncodingError
import time
import os
from tqdm import tqdm
wsdl = "https://www.brenda-enzymes.org/soap/brenda_zeep.wsdl"
email = "1055285901@qq.com"
password = hashlib.sha256("LBXSQJLRTZ1124".encode("utf-8")).hexdigest()
# 创建SOAP客户端
client = Client(wsdl)
# 查询所有 EC numbers
parameters_ec = (email, password)
ECnumbers = client.service.getEcNumbersFromEcNumber(*parameters_ec)
print(f"Retrieved {len(ECnumbers)} EC numbers.")

# 初始化存储数据的列表
data = []

# 加载已存在的数据（如果有）
'''
在重新运行时，可以注释掉这部分代码，以避免遗漏数据
'''
# if os.path.exists(output_file):
#     df_existing = pd.read_csv(output_file)
#     processed_ec_numbers = set(df_existing['EC_number'].unique())
#     data = df_existing.to_dict('records')
# else:
#     processed_ec_numbers = set()



def fetch_data(client, email, password, ecNumber, service_method, max_retries=5):
    for attempt in range(max_retries):
        try:
            if service_method == "kcat":
                parameters = (email, password, f"ecNumber*{ecNumber}", "turnoverNumber*", "turnoverNumberMaximum*", "substrate*", "commentary*", "organism*", "ligandStructureId*", "literature*")
                return client.service.getTurnoverNumber(*parameters)
            elif service_method == "km":
                parameters = (email, password, f"ecNumber*{ecNumber}", "kmValue*", "kmValueMaximum*", "substrate*", "commentary*", "organism*", "ligandStructureId*", "literature*")
                return client.service.getKmValue(*parameters)
        except (ConnectionError, ChunkedEncodingError, Fault, TransportError) as e:
            # print(f"Attempt {attempt + 1} failed for EC {ecNumber}: {e}")
            if attempt == max_retries - 1:
                print('error', ecNumber , service_method)
            time.sleep(2*attempt)
    return None

# 遍历 EC numbers 列表并提取数据
for ecNumber in tqdm(ECnumbers):
    '''同上，由于数据量较大，可以注释掉这部分代码，以避免遗漏数据'''
    # if ecNumber in processed_ec_numbers:
    #     print(f"EC number {ecNumber} 已处理，跳过...")
    #     continue
    kcat_results = fetch_data(client, email, password, ecNumber, "kcat")
    km_results = fetch_data(client, email, password, ecNumber, "km")

    # 处理并存储 kcat 数据
    if kcat_results:
        for result in kcat_results:
            data.append({
                "EC_number": ecNumber,
                "Type": "kcat",
                "Value": getattr(result, 'turnoverNumber', ''),
                "Maximum": getattr(result, 'turnoverNumberMaximum', ''),
                "Substrate": getattr(result, 'substrate', ''),
                "Commentary": getattr(result, 'commentary', ''),
                "Organism": getattr(result, 'organism', ''),
                "LigandStructureId": getattr(result, 'ligandStructureId', ''),
                "Literature": getattr(result, 'literature', '')
            })
    # 处理并存储 km 数据
    if km_results:
        for result in km_results:
            data.append({
                "EC_number": ecNumber,
                "Type": "km",
                "Value": getattr(result, 'kmValue', ''),
                "Maximum": getattr(result, 'kmValueMaximum', ''),
                "Substrate": getattr(result, 'substrate', ''),
                "Commentary": getattr(result, 'commentary', ''),
                "Organism": getattr(result, 'organism', ''),
                "LigandStructureId": getattr(result, 'ligandStructureId', ''),
                "Literature": getattr(result, 'literature', '')
            })

    # 每次处理完一个EC number后，保存数据到文件
    df = pd.DataFrame(data)
    df.to_csv(output_file, index=False)
    # print(f"EC number {ecNumber} 的数据已保存。")

print("所有数据提取并保存完成")# 8424 6h 20240814
'''error 1.6.5.5
 29%|██▊       | 2418/8424 [1:05:58<3:37:58,  2.18s/it] 
error 1.7.1.B3
 48%|████▊     | 4057/8424 [1:56:01<5:35:18,  4.61s/it] 
error 2.5.1.62
 63%|██████▎   | 5344/8424 [2:57:25<2:43:53,  3.19s/it] 
error 3.1.8.1 #不应该有
 88%|████████▊ | 7413/8424 [5:12:53<39:34,  2.35s/it]   
error 4.2.3.23
 92%|█████████▏| 7791/8424 [5:33:16<23:57,  2.27s/it]  
error 5.2.1.8
 96%|█████████▌| 8047/8424 [5:47:25<46:40,  7.43s/it]  
error 6.1.1.12
error 6.1.1.12'''


Retrieved 8424 EC numbers.


 28%|██▊       | 2364/8424 [1:04:04<5:08:10,  3.05s/it] 

error 1.6.5.5


 29%|██▊       | 2418/8424 [1:05:58<3:37:58,  2.18s/it] 

error 1.7.1.B3


...

error 5.2.1.8


 96%|█████████▌| 8047/8424 [5:47:25<46:40,  7.43s/it]  

error 6.1.1.12
error 6.1.1.12


100%|██████████| 8424/8424 [6:09:46<00:00,  2.63s/it]  

所有数据提取并保存完成


对未正确提取的ec号做二次提取

In [1]:
output_file = 'kcat_km_values.csv'

import hashlib
import pandas as pd
from zeep import Client
from zeep.exceptions import Fault, TransportError
from requests.exceptions import ConnectionError, ChunkedEncodingError
import time
import os
from tqdm import tqdm
wsdl = "https://www.brenda-enzymes.org/soap/brenda_zeep.wsdl"
email = "1055285901@qq.com"
password = hashlib.sha256("LBXSQJLRTZ1124".encode("utf-8")).hexdigest()
# 创建SOAP客户端
client = Client(wsdl)
# 查询所有 EC numbers
parameters_ec = (email, password)
ECnumbers = client.service.getEcNumbersFromEcNumber(*parameters_ec)
print(f"Retrieved {len(ECnumbers)} EC numbers.")

# 初始化存储数据的列表
data = []

def fetch_data(client, email, password, ecNumber, service_method, max_retries=5):
    for attempt in range(max_retries):
        try:
            if service_method == "kcat":
                parameters = (email, password, f"ecNumber*{ecNumber}", "turnoverNumber*", "turnoverNumberMaximum*", "substrate*", "commentary*", "organism*", "ligandStructureId*", "literature*")
                return client.service.getTurnoverNumber(*parameters)
            elif service_method == "km":
                parameters = (email, password, f"ecNumber*{ecNumber}", "kmValue*", "kmValueMaximum*", "substrate*", "commentary*", "organism*", "ligandStructureId*", "literature*")
                return client.service.getKmValue(*parameters)
        except (ConnectionError, ChunkedEncodingError, Fault, TransportError) as e:
            # print(f"Attempt {attempt + 1} failed for EC {ecNumber}: {e}")
            if attempt == max_retries - 1:
                print('error', ecNumber, service_method)
                error_type = type(e).__name__
                print(f"Attempt {attempt + 1} failed for EC {ecNumber}: [{error_type}] {e}")
            time.sleep(2*attempt)  # 指数回退重试
    return None
failed_ec_numbers = [
    "1.6.5.5","1.7.1.B3","2.5.1.62","3.1.8.1","4.2.3.23","5.2.1.8","6.1.1.12"
    ]# kcat只有6.1.1.12

for ecNumber in tqdm(failed_ec_numbers):

    kcat_results = fetch_data(client, email, password, ecNumber, "kcat")
    km_results = fetch_data(client, email, password, ecNumber, "km")
    # 处理并存储 kcat 数据
    if kcat_results:
        for result in kcat_results:
            data.append({
                "EC_number": ecNumber,
                "Type": "kcat",
                "Value": getattr(result, 'turnoverNumber', ''),
                "Maximum": getattr(result, 'turnoverNumberMaximum', ''),
                "Substrate": getattr(result, 'substrate', ''),
                "Commentary": getattr(result, 'commentary', ''),
                "Organism": getattr(result, 'organism', ''),
                "LigandStructureId": getattr(result, 'ligandStructureId', ''),
                "Literature": getattr(result, 'literature', '')
            })
    # 处理并存储 km 数据
    if km_results:
        for result in km_results:
            data.append({
                "EC_number": ecNumber,
                "Type": "km",
                "Value": getattr(result, 'kmValue', ''),
                "Maximum": getattr(result, 'kmValueMaximum', ''),
                "Substrate": getattr(result, 'substrate', ''),
                "Commentary": getattr(result, 'commentary', ''),
                "Organism": getattr(result, 'organism', ''),
                "LigandStructureId": getattr(result, 'ligandStructureId', ''),
                "Literature": getattr(result, 'literature', '')
            })

    # 保存数据到文件
    df2 = pd.DataFrame(data)
    output_file = 'kcat_km_values.csv'
    df1 = pd.read_csv(output_file)
    df = pd.concat([df1, df2], ignore_index=True)
    df.to_csv(output_file, index=False)


Retrieved 8424 EC numbers.


  0%|          | 0/7 [00:00<?, ?it/s]

error 1.6.5.5 km
Attempt 5 failed for EC 1.6.5.5: [TransportError] Server returned response (200) with invalid XML: Invalid XML content received (PCDATA invalid Char value 2, line 2, column 53699).
Content: b'<?xml version="1.0" encoding="UTF-8"?>\n<SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:ns1="ws.brenda" xmlns:SOAP-ENC="http://schemas.xmlsoap.org/soap/encoding/" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" SOAP-ENV:encodingStyle="http://schemas.xmlsoap.org/soap/encoding/"><SOAP-ENV:Body><ns1:getKmValueResponse><return SOAP-ENC:arrayType="ns1:kmValueObject[95]" xsi:type="ns1:ArrayOfKmValues"><item xsi:type="ns1:kmValueObject"><literature SOAP-ENC:arrayType="xsd:int[1]" xsi:type="ns1:ArrayOfIntegers"><item xsi:type="xsd:int">697701</item></literature><substrate xsi:type="xsd:string">NADPH</substrate><kmValue xsi:type="xsd:string">0.0025</kmValue><kmValueMaximum xsi:type="xsd:string"></kmValue

 14%|█▍        | 1/7 [00:29<02:59, 29.91s/it]

error 1.7.1.B3 km
Attempt 5 failed for EC 1.7.1.B3: [TransportError] Server returned response (200) with invalid XML: Invalid XML content received (PCDATA invalid Char value 2, line 2, column 23574).
Content: b'<?xml version="1.0" encoding="UTF-8"?>\n<SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:ns1="ws.brenda" xmlns:SOAP-ENC="http://schemas.xmlsoap.org/soap/encoding/" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" SOAP-ENV:encodingStyle="http://schemas.xmlsoap.org/soap/encoding/"><SOAP-ENV:Body><ns1:getKmValueResponse><return SOAP-ENC:arrayType="ns1:kmValueObject[40]" xsi:type="ns1:ArrayOfKmValues"><item xsi:type="ns1:kmValueObject"><literature SOAP-ENC:arrayType="xsd:int[1]" xsi:type="ns1:ArrayOfIntegers"><item xsi:type="xsd:int">742109</item></literature><substrate xsi:type="xsd:string">NADPH</substrate><kmValue xsi:type="xsd:string">0.00085</kmValue><kmValueMaximum xsi:type="xsd:string"></kmVa

...

 71%|███████▏  | 5/7 [02:08<00:49, 24.83s/it]

error 5.2.1.8 km
Attempt 5 failed for EC 5.2.1.8: [TransportError] Server returned response (200) with invalid XML: Invalid XML content received (PCDATA invalid Char value 2, line 2, column 5002).
Content: b'<?xml version="1.0" encoding="UTF-8"?>\n<SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:ns1="ws.brenda" xmlns:SOAP-ENC="http://schemas.xmlsoap.org/soap/encoding/" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" SOAP-ENV:encodingStyle="http://schemas.xmlsoap.org/soap/encoding/"><SOAP-ENV:Body><ns1:getKmValueResponse><return SOAP-ENC:arrayType="ns1:kmValueObject[23]" xsi:type="ns1:ArrayOfKmValues"><item xsi:type="ns1:kmValueObject"><literature SOAP-ENC:arrayType="xsd:int[1]" xsi:type="ns1:ArrayOfIntegers"><item xsi:type="xsd:int">2527</item></literature><substrate xsi:type="xsd:string">succinyl-Ala-Ala-Pro-Phe 4-nitroanilide</substrate><kmValue xsi:type="xsd:string">0.451</kmValue><kmValueMaximum x

 86%|████████▌ | 6/7 [02:34<00:25, 25.36s/it]

error 6.1.1.12 kcat
Attempt 5 failed for EC 6.1.1.12: [TransportError] Server returned response (200) with invalid XML: Invalid XML content received (PCDATA invalid Char value 1, line 2, column 9467).
Content: b'<?xml version="1.0" encoding="UTF-8"?>\n<SOAP-ENV:Envelope xmlns:SOAP-ENV="http://schemas.xmlsoap.org/soap/envelope/" xmlns:ns1="ws.brenda" xmlns:SOAP-ENC="http://schemas.xmlsoap.org/soap/encoding/" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" SOAP-ENV:encodingStyle="http://schemas.xmlsoap.org/soap/encoding/"><SOAP-ENV:Body><ns1:getTurnoverNumberResponse><return SOAP-ENC:arrayType="ns1:turnoverNumber[49]" xsi:type="ns1:ArrayOfTurnoverNumbers"><item xsi:type="ns1:turnoverNumber"><literature SOAP-ENC:arrayType="xsd:int[1]" xsi:type="ns1:ArrayOfIntegers"><item xsi:type="xsd:int">704504</item></literature><substrate xsi:type="xsd:string">ATP</substrate><turnoverNumberMaximum xsi:type="xsd:string"></turnoverNumberMaximum><comment

100%|██████████| 7/7 [03:27<00:00, 29.67s/it]


### devide the kcat and km
### if exist max: value=max

In [1]:
######################################################################
# devide the kcat and km
# max replace value

import pandas as pd
import re
from tqdm import tqdm
input_file = 'kcat_km_values.csv'
output_file1 = 'kcat_data.csv'
output_file2 ='km_data.csv'
data = pd.read_csv(input_file)
data.drop(columns=['LigandStructureId', 'Literature'], inplace=True)
if 'Commentary' in data.columns:
    data.rename(columns={'Commentary': 'enzymeType'}, inplace=True)
# 初始化两个空的DataFrame，用于存储不同类型的数据
data_kcat = pd.DataFrame(columns=data.columns)
data_km = pd.DataFrame(columns=data.columns)
# 遍历数据行
for index, row in tqdm(data.iterrows()):
    if row['Value'] <= 0:
        continue  # 如果Value小于0，则跳过此行
    desc = str(row['enzymeType']).lower()
    if 'mutant' in desc or 'mutated' in desc:
        mutant = re.findall(r'[A-Z]\d+[A-Z]', desc)  # 查找所有变异
        if len(mutant) >= 1:  # 如果存在多个变异,divide
            enzymeType = '/'.join(mutant)
        else:
            continue
    else:
        enzymeType = 'wildtype'

    row['enzymeType'] = enzymeType  # 用enzymeType替换Commentary
    if pd.notnull(row['Maximum']) and row['Maximum'] != '':
        row['Value'] = row['Maximum']  # 如果Maximum存在，替换Value
    # 根据Type字段分配到不同的DataFrame
    if row['Type'].lower() == 'kcat':
        data_kcat = pd.concat([data_kcat, pd.DataFrame([row])], ignore_index=True)
    elif row['Type'].lower() == 'km':
        data_km = pd.concat([data_km, pd.DataFrame([row])], ignore_index=True)
data_kcat.drop(columns=['Maximum'], inplace=True)
data_km.drop(columns=['Maximum'], inplace=True)
print(len(data_kcat), len(data_km))
data_kcat.to_csv(output_file1, index=False)# 56030 #去重后为37645
data_km.to_csv(output_file2, index=False) # 129142

0it [00:00, ?it/s]/tmp/ipykernel_805758/3779252579.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data_kcat = pd.concat([data_kcat, pd.DataFrame([row])], ignore_index=True)
541it [00:00, 1858.59it/s]/tmp/ipykernel_805758/3779252579.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data_km = pd.concat([data_km, pd.DataFrame([row])], ignore_index=True)
262831it [05:13, 837.39it/s] 


In [4]:
import pandas as pd
df = pd.read_csv('kcat_data.csv')
df_unique = df.drop_duplicates(subset=df.columns.difference(['Value']))
df_unique.to_csv('kcat_data_clean.csv', index=False)
print(len(df), len(df_unique))# 56030 37645

56030 37645


brenda smiles

In [3]:
import json
import requests
import multiprocessing as mp
import time
import random
from multiprocessing import Pool
from tqdm import tqdm
import pandas as pd
import pubchempy as pcp
# from pubchempy import Compound, get_compounds
'''
obtain canonical SMILES just by chemical name using PubChem API
obtain SMILES by PubChem API using the website
使用PubChem API根据化学品名称获取它们的标准SMILES表示。该脚本利用Python的多进程能力来加速API请求。
'''
# name_smiles = dict()
# def get_smiles(name):
#     max_retries = 3
#     for attempt in range(max_retries):
#         try:
#             url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/%s/property/CanonicalSMILES/TXT' % name
#             req = requests.get(url)
#             if req.status_code == 200:
#                 smiles = req.content.splitlines()[0].decode()
#                 name_smiles[name] = smiles
#                 return
#             else:
#                 print(url)
#                 print(req)
#                 print(req.status_code)
#                 time.sleep(random.uniform(1, 3))  # Adding a random delay
#         except Exception as e:
#             time.sleep(random.uniform(1, 3))  # Adding a random delay
#     name_smiles[name] = None
# def process_items(names):
#     with Pool(4) as pool:
#         for _ in tqdm(pool.imap_unordered(get_smiles, names), total=len(names)):
#             time.sleep(random.uniform(0.1, 0.3))  # Adding a short delay between tasks to avoid triggering anti-scraping measures
name_smiles = dict()

def main():
    df_1 = pd.read_csv('kcat_data.csv')
    # df_2 = pd.read_csv('km_data.csv')
    substrates1 = df_1['Substrate'].tolist()
    # substrates2 = df_2['Substrate'].tolist()
    # substrates = substrates1 + substrates2
    substrates = substrates1
    names = list(set(substrates))
    print("Number of unique substrates:", len(names))
    ########################################################
    for compound_name in tqdm(names):
        compounds = pcp.get_compounds(compound_name, 'name')
        if compounds:
            smiles = compounds[0].canonical_smiles
            name_smiles[compound_name] = smiles
        else:
            name_smiles[compound_name] = None
    ########################################################
    with open('Kcat_data_smiles.json', 'w') as wf:
        json.dump(name_smiles, wf, indent=2)
if __name__ == '__main__':
    main()
# 得到的数据保存在Kcat_data_smiles.json文件中，仍有许多(2/3)化合物没有找到对应的SMILES表示。
# 运行时间6.5小时 20240815 未找到: 9786/16142 找到6000多

Number of unique substrates: 16142


  0%|          | 0/10 [00:00<?, ?it/s]

https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Abz-Tyr-Tyr-Abu-(5-amino-2-nitrobenzoyl)-Pro-NH2/property/CanonicalSMILES/TXT


 10%|█         | 1/10 [00:00<00:08,  1.05it/s]

https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/3-carboxyphenyl N-(phenylacetyl)-alpha-serinate/property/CanonicalSMILES/TXT


...

https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Abz-AGRK-SLTnY-amide/property/CanonicalSMILES/TXT


 90%|█████████ | 9/10 [00:02<00:00,  3.73it/s]

https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/(S)-solketal aldehyde/property/CanonicalSMILES/TXT


100%|██████████| 10/10 [00:03<00:00,  3.16it/s]

Number of unique substrates with SMILES: 0
Number of compounds without SMILES: 16142


In [4]:
import json

# Read the JSON file
with open('Kcat_data_smiles.json', 'r') as file:
    data = json.load(file)

# Count the number of empty values
count = sum([1 for value in data.values()])
print(count)
empty_count = sum([1 for value in data.values() if value is None])

empty_count

16142


9786

In [3]:
import json

# Read the JSON file
with open('../complementaryData/Kcat_brenda_smiles.json', 'r') as file:
    data = json.load(file)

# Count the number of empty values
count = sum([1 for value in data.values()])
print(count)
empty_count = sum([1 for value in data.values() if value is None])

empty_count

15174


9020

### SABIO-RK DATA
(数据来源enzyme.dat是20240327 现更新为20240724)

In [5]:
#!/usr/bin/python
# coding: utf-8


import requests
import time
from tqdm import tqdm
import random
# import os
# Extract EC number list from ExPASy, which is a repository of information relative to the nomenclature of enzymes.You can get the enzyme data file below at https://ftp.expasy.org/databases/enzyme/

def eclist():# get all EC numbers from the data file as a list
    with open('../../Data/EC_enzyme/enzyme.dat', 'r') as rf :
        lines = rf.readlines()

    ec_list = list()
    for line in lines :
        if line.startswith('ID') :# example:ID   1.1.1.1
            ec = line.strip().split('  ')[1]
            ec_list.append(ec)
    # print(ec_list)
    # print(len(ec_list)) # 8328
    return ec_list

def sabio_info(allEC):# download data from sabio (including not only kcat)
    QUERY_URL = 'http://sabiork.h-its.org/sabioRestWebServices/kineticlawsExportTsv'

    # specify search fields and search terms

    # query_dict = {"ECNumber":'"1.1.1.1"',}
    i = 0
    count = 0
    for EC in tqdm(allEC) :
        EC = EC.strip() # EC contains a space in front
        # print(EC)
        i += 1
        # print('This is %d ----------------------------' %i)
        # print('Downloading '+EC)
        query_dict = {"ECNumber":'%s' %EC,}
        query_string = ' AND '.join(['%s:%s' % (k,v) for k,v in query_dict.items()])

        # specify output fields and send request

        query = {'fields[]':['EntryID', 'Substrate', 'EnzymeType', 'PubMedID', 'Organism', 'UniprotID','ECNumber','Parameter'], 'q':query_string}

        request = requests.post(QUERY_URL, params = query)
        time.sleep(random.random()*3) # the sleep time here maybe a little long

        if request.text :
            with open('../../Data/database/Kcat_sabio_4/%s.txt' %EC, 'w',encoding='utf-8') as ECfile :
                ECfile.write(request.text)
        else:
            count+=1
    print("there is %d not correctly download" %count)


if __name__ == '__main__' :
    sabio_info(eclist()) # 8328 [10:11:14<00:00,  4.45s/it] succesfully run on linux # 8:30:14 20240728
                         # there is 0 not correctly download

  0%|          | 0/8328 [00:00<?, ?it/s]

In [2]:
#!/usr/bin/python
# coding: utf-8
'''
从多个TXT(TSV)文件中读取Kcat和Km 并将特定条件下的数据集合到一个新的TSV文件中。
example:
EntryID	Type	ECNumber	Substrate	EnzymeType	PubMedID	Organism	UniprotID	Value	Unit
1	kcat	3.2.1.103	Gal-beta1->4GlcNAc-beta1->3Gal-beta1->4GlcNAc-beta-pNP	wildtype	12950254	Citrobacter freundii		72.9	s^(-1)
EntryID	Substrate	EnzymeType	PubMedID	Organism	UniprotID	ECNumber	parameter.type	parameter.associatedSpecies	parameter.startValue	parameter.endValue	parameter.standardDeviation	parameter.unit
3804	1-Octanol;NAD+	wildtype class III	2936344	Homo sapiens	P11766	1.1.1.1	Km	1-Octanol	5.5E-4		-	M
'''
# goal:
# EC_number	Type	Value	Maximum	Substrate	Commentary	Organism	LigandStructureId	Literature

import os
import csv
from tqdm import tqdm

with open("../../Data/database/Kcat_sabio_4_unisubstrate.tsv", 'w') as wf:
    # with open("./Kcat_sabio.tsv", "wt") as outfile :
    tsv_writer = csv.writer(wf, delimiter="\t")
    tsv_writer.writerow(["EntryID", "Type", "ECNumber", "Substrate", "EnzymeType", "PubMedID", 
        "Organism", "UniprotID", "Value", "Unit"])
    
    filenames = os.listdir('../../Data/database/Kcat_sabio_4')
    print('len(filenames)',len(filenames)) # 1741 EC files --8236
    i = 0
    j = 0
    for filename in tqdm(filenames) :
        if filename != '.DS_Store' :
            with open("../../Data/database/Kcat_sabio_4/%s" % filename, 'r', encoding="utf-8") as rf :
                lines = rf.readlines()
                # print(lines)
            for line in lines[1:] :# not include the head of table
                data = line.strip().split('\t')
                # print(data)
                try :
                    if data[7] == 'kcat' and data[9] :
                        i += 1
                        # print(i)
                        # print(data)
                        entryID = data[0]
                        for line in lines[1:] :
                            data2 = line.strip().split('\t')
                            if data2[0] == entryID and data2[7] == 'Km' :# same entryID and have both kcat and km to get the substrate
                                j += 1
                                # print(j)
                                #tsv_writer.writerow(["EntryID", "Type", "ECNumber", "Substrate", "EnzymeType", "PubMedID", "Organism", "UniprotID", "Value", "Unit"]) yi yi dui ying, substrate = data2[8]
                                tsv_writer.writerow([j, data[7], data[6], data2[8], data[2], data[3], data[4], data[5], data[9], data[-1]])
                except :
                    continue
    # tsv_writer.writerow(["EntryID", "Type", "ECNumber", "Substrate", "EnzymeType", "PubMedID", 
    #     "Organism", "UniprotID", "Value", "Unit"])

8236


100%|██████████| 8236/8236 [00:24<00:00, 336.60it/s] 


sabio3

In [3]:
#!/usr/bin/python
# coding: utf-8

# Run in python 3.7
'''
这段Python代码主要功能是处理一个TSV 代码的目的是清洗这些数据，选择最大的Kcat值，统一单位，并去除重复项，最终将清洗后的数据保存到一个新的TSV文件中。

output example:
Type	ECNumber	Substrate	EnzymeType	PubMedID	Organism	UniprotID	Value	Unit
kcat	3.2.1.103	Gal-beta1->4GlcNAc-beta1->3Gal-beta1->4GlcNAc-beta-pNP	wildtype	12950254	Citrobacter freundii		72.9	s^(-1)
'''
import csv

with open("../../Data/database/Kcat_sabio_4_unisubstrate.tsv", "r", encoding='utf-8') as rf :
    # lines = file.readlines()[1:].strip('\n')
    lines = rf.readlines()[1:]
    lines = [line for line in lines if line.strip()!='']
    # i=0
    Kcat_data = list()
    Kcat_data_include_value = list()
    for line in lines:
        data = line.strip().split('\t')
        Type = data[1]
        ECNumber = data[2]
        Substrate = data[3]
        EnzymeType = data[4]
        PubMedID = data[5]
        Organism = data[6]
        UniprotID = data[7]
        Value = data[8]
        Unit = data[9]
        Kcat_data_include_value.append([Type, ECNumber, Substrate, EnzymeType, PubMedID, Organism, UniprotID, Value, Unit])
        Kcat_data.append([Type, ECNumber, Substrate, EnzymeType, PubMedID, Organism, UniprotID])

print("len(Kcat_data)",len(Kcat_data))  # 22683 items for not unique substrate --26345

# 去除重复项
new_lines = list()
for line in Kcat_data :
    if line not in new_lines :
        new_lines.append(line)

print("len(new_lines)",len(new_lines))  # 21627 included all elements, 18296 included all except for Kcat value and unit --21507

# i = 0
clean_Kcat = list()
for new_line in new_lines :
    value_unit = dict()
    Kcat_values = list()
    for line in Kcat_data_include_value :
        if line[:-2] == new_line :
            value = line[-2]
            value_unit[str(float(value))] = line[-1]
            # print(type(value))  # <class 'str'>
            Kcat_values.append(float(value))
    # print(value_unit)
    # print(Kcat_values)
    max_value = max(Kcat_values) # choose the maximum one for duplication Kcat value under the same entry as the data what we use
    unit = value_unit[str(max_value)]
    # print(max_value)
    # print(unit)

    if unit in ['mol*s^(-1)*mol^(-1)', 's^(-', '-'] :
        unit = 's^(-1)'
        # print("unit changed")# 29 unit changed in total
    new_line.append(str(max_value))
    new_line.append(unit)
    if new_line[-1] == 's^(-1)' :
        clean_Kcat.append(new_line)


# print(clean_Kcat)
print("len(clean_Kcat)",len(clean_Kcat))  # 18243 after unifing the Kcat value unit to 's^(-1)', in which 16825 has a specific Unipro ID # 21461 --21452

with open("../../Data/database/Kcat_sabio_clean_unisubstrate.tsv", "w") as wf :
    records = ['Type', 'ECNumber', 'Substrate', 'EnzymeType', 'PubMedID', 'Organism', 'UniprotID', 'Value', 'Unit']
    wf.write('\t'.join(records) + '\n')
    for line in clean_Kcat :
        wf.write('\t'.join(line) + '\n')


Kcat_data 26345
new_lines 21507
clean_Kcat 21452


sabio smiles

In [4]:
import json
import requests
import multiprocessing as mp
import time
import random
from multiprocessing import Pool
from tqdm import tqdm
import pandas as pd
import pubchempy as pcp
'''
obtain canonical SMILES just by chemical name using PubChem API
obtain SMILES by PubChem API using the website
使用PubChem API根据化学品名称获取它们的标准SMILES表示。该脚本利用Python的多进程能力来加速API请求。
'''
name_smiles = dict()
def main():
    with open("../../Data/database/Kcat_sabio_clean_unisubstrate.tsv", "r", encoding='utf-8') as rf:
        lines = rf.readlines()[1:]

    substrates = [line.strip().split('\t')[2] for line in lines]
    names = list(set(substrates))
    print("Number of unique substrates:", len(names))
    for compound_name in tqdm(names):
        compounds = pcp.get_compounds(compound_name, 'name')
        if compounds:
            smiles = compounds[0].canonical_smiles
            name_smiles[compound_name] = smiles
        else:
            name_smiles[compound_name] = None
    with open('../../Data/database/Kcat_sabio_smiles.json', 'w') as wf:
        json.dump(name_smiles, wf, indent=2)
if __name__ == '__main__':
    main()


Number of unique substrates: 3274


100%|██████████| 3274/3274 [16:57<00:00,  3.22it/s]


### Combination

In [1]:
# This script is to combine the Kcat data from BRENDA and Sabio-RK databases
# Units are all s^(-1)

import json
import pandas as pd
from tqdm import tqdm

# Load SMILES mappings from JSON files
with open('../../Data/database/Kcat_sabio_smiles.json', 'r') as infile1:
    sabio_name_smiles = json.load(infile1)

with open('Kcat_km_data_smiles.json', 'r') as infile2:
    brenda_name_smiles = json.load(infile2)

# Load BRENDA and SABIO data using pandas
brenda_df = pd.read_csv("kcat_data.csv")# 56243
sabio_df = pd.read_csv("../../Data/database/Kcat_sabio_clean_unisubstrate.tsv", sep='\t')# 21453

# Initialize dictionaries and lists to store processed data
Substrate_name = {}
Substrate_smiles = {}
entry_uniprot = {}
Kcat_data = []
Kcat_data_include_value = []

# Process BRENDA data
print("Processing BRENDA data...")
not_found_brenda = 0
for _, row in brenda_df.iterrows():
    ECNumber = row['EC_number']
    Substrate = row['Substrate']
    EnzymeType = tuple(row['enzymeType'].split('/'))
    Organism = row['Organism']
    Value = row['Value']

    smiles = brenda_name_smiles.get(Substrate)
    if smiles is not None:
        Substrate_name[Substrate.lower()] = Substrate
        Substrate_smiles[Substrate.lower() + '&smiles'] = smiles
        Kcat_data_include_value.append([ECNumber, Substrate.lower(), EnzymeType, Organism, Value])
        Kcat_data.append([ECNumber, Substrate.lower(), EnzymeType, Organism])
    else:
        not_found_brenda+=1
print('not found brenda:', not_found_brenda, ' ', len(brenda_df))
# Process SABIO data
print("Processing SABIO data...")
not_found_sabio = 0
for _, row in sabio_df.iterrows():
    ECNumber = row['ECNumber']
    Substrate = row['Substrate']
    EnzymeType = tuple(row['EnzymeType'].split('/'))
    Organism = row['Organism']
    UniprotID = row['UniprotID']
    Value = row['Value']

    smiles = sabio_name_smiles.get(Substrate)
    if smiles is not None:
        Substrate_name[Substrate.lower()] = Substrate
        Substrate_smiles[Substrate.lower() + '&smiles'] = smiles
        entry_uniprot[ECNumber + Substrate.lower() + Organism] = UniprotID
        Kcat_data_include_value.append([ECNumber, Substrate.lower(), EnzymeType, Organism, Value])
        Kcat_data.append([ECNumber, Substrate.lower(), EnzymeType, Organism])
    else:
        not_found_sabio += 1
print('not found sabio :', not_found_sabio,' ',len(sabio_df))
# Remove duplicates
Kcat_data = list(map(list, set(map(tuple, Kcat_data))))
print('len(Kcat_data)',len(Kcat_data))
# Create a clean Kcat list by selecting the maximum Kcat value for each unique entry
clean_Kcat = []
print("Selecting maximum Kcat values...")
for entry in tqdm(Kcat_data):
    ECNumber, Substrate, EnzymeType, Organism = entry
    values = [float(v[4]) for v in Kcat_data_include_value if v[:-1] == entry]
    max_value = max(values)
    Substrate_name_clean = Substrate_name[Substrate]
    Smiles = Substrate_smiles[Substrate + '&smiles']
    UniprotID = entry_uniprot.get(ECNumber + Substrate + Organism, '')
    clean_entry = [ECNumber, Substrate_name_clean, '/'.join(EnzymeType), Organism, Smiles, UniprotID, str(max_value)]
    clean_Kcat.append(clean_entry)

# Save the clean Kcat data to a TSV file
clean_kcat_df = pd.DataFrame(clean_Kcat, columns=['ECNumber', 'Substrate', 'EnzymeType', 'Organism', 'Smiles', 'UniprotID', 'Value'])
clean_kcat_df.to_csv("../../Data/database/Kcat_combination_0730.tsv", sep='\t', index=False)

# Load the combined data again to remove duplicates by SMILES
clean_kcat_df = pd.read_csv("../../Data/database/Kcat_combination_0730.tsv", sep='\t')

# Initialize lists to store the final clean data
Kcat_data = []
Kcat_data_include_value = []

# Process the combined data to remove duplicates by SMILES
print("Removing duplicates by SMILES...")
for _, row in clean_kcat_df.iterrows():
    ECNumber = row['ECNumber']
    Substrate = row['Substrate']
    EnzymeType = tuple(row['EnzymeType'].split('/'))
    Organism = row['Organism']
    Smiles = row['Smiles']
    UniprotID = row['UniprotID']
    Value = row['Value']

    Substrate_name[Smiles] = Substrate
    entry_uniprot[ECNumber + Smiles + Organism] = UniprotID
    Kcat_data_include_value.append([ECNumber, EnzymeType, Organism, Smiles, Value])
    Kcat_data.append([ECNumber, EnzymeType, Organism, Smiles])

# Remove duplicates
Kcat_data = list(map(list, set(map(tuple, Kcat_data))))

# Create the final clean Kcat list by selecting the maximum Kcat value for each unique entry
clean_Kcat = []
print("Selecting maximum Kcat values...")
for entry in tqdm(Kcat_data):
    ECNumber, EnzymeType, Organism, Smiles = entry
    values = [float(v[4]) for v in Kcat_data_include_value if v[:-1] == entry]
    max_value = max(values)
    Substrate_name_clean = Substrate_name[Smiles]
    UniprotID = entry_uniprot.get(ECNumber + Smiles + Organism, '')
    clean_entry = [ECNumber, '/'.join(EnzymeType), Organism, Smiles, Substrate_name_clean, UniprotID, str(max_value)]
    clean_Kcat.append(clean_entry)

# Save the final clean Kcat data to a TSV file
final_clean_kcat_df = pd.DataFrame(clean_Kcat, columns=['ECNumber', 'EnzymeType', 'Organism', 'Smiles', 'Substrate', 'UniprotID', 'Value'])
print(len(final_clean_kcat_df))
final_clean_kcat_df.to_csv("../../Data/database/Kcat_combination_0731_st.tsv", sep='\t', index=False)

print("Data cleaning and combination completed.")
'''Processing BRENDA data...
not found brenda: 17597   56242
Processing SABIO data...
not found sabio : 3100   21452
len(Kcat_data) 40619
Selecting maximum Kcat values...
100%|██████████| 40619/40619 [04:09<00:00, 162.97it/s]
Removing duplicates by SMILES...
Selecting maximum Kcat values...
100%|██████████| 38335/38335 [03:07<00:00, 204.14it/s]
38335
Data cleaning and combination completed.'''

Processing BRENDA data...
not found brenda: 17597   56242
Processing SABIO data...
not found sabio : 3100   21452
len(Kcat_data) 40619
Selecting maximum Kcat values...


  2%|▏         | 626/40619 [00:03<04:13, 157.50it/s]

### Plan1:从uniprot拿id和seq

In [3]:
# import json
# import pandas as pd
# import requests
# from urllib import request

# def seq_by_ec_organism(ec_number, organism):
#     IdSeq = dict()
#     base_url = "https://rest.uniprot.org/uniprotkb/search"
#     query = f"ec:{ec_number} AND organism_name:\"{organism}\" AND reviewed:true"
#     params = {
#         "query": query,
#         "format": "fasta",
#         "size": 1  # 获取第一个匹配的序列
#     }
    
#     response = requests.get(base_url, params=params)
#     try:
#         print(response.status_code)
#         if response.status_code == 200:
#             respdata = response.text
#             # print(respdata)
#             seq = dict()
#             for line in respdata.split('\n') :
#                 if line.startswith('>') :
#                     name=line
#                     seq[name] = ''
#                 else :
#                     seq[name] += line.replace('\n', '').strip()
#             IdSeq[ec_number+'&'+organism] =  list(seq.values())
#     except:
#         print(ec_number+'&'+organism, "can not find from uniprot!")
#         IdSeq[ec_number+'&'+organism] = None
#     return IdSeq[ec_number+'&'+organism]

# def get_uniprot_sequence(uniprot_id):
#     """Retrieve protein sequence from Uniprot given a Uniprot ID"""
#     url = f"https://www.uniprot.org/uniprot/{uniprot_id}.fasta"
#     try:
#         data = request.urlopen(url)
#         respdata = data.read().decode("utf-8").strip()
#         sequence = "".join(respdata.split("\n")[1:])
#         return sequence
#     except Exception as e:
#         print(f"Error retrieving sequence for Uniprot ID {uniprot_id}: {e}")
#         return None

# # Read the Kcat data
# kcat_df = pd.read_csv("../../Data/database/Kcat_combination_0731_st.tsv", sep='\t')

# # Initialize dictionaries to store Uniprot IDs and sequences
# uniprot_ids = {}
# uniprot_sequences = {}

# # Retrieve Uniprot IDs based on ECNumber and Organism
# for index, row in kcat_df.iterrows():
#     ec_number = row['ECNumber']
#     organism = row['Organism']
#     if pd.isna(row['UniprotID']):
#         uniprot_id = seq_by_ec_organism(ec_number, organism)
#         if uniprot_id:
#             uniprot_ids[(ec_number, organism)] = uniprot_id
#         else:
#             uniprot_ids[(ec_number, organism)] = None

# # Add retrieved Uniprot IDs to the dataframe
# kcat_df['UniprotID'] = kcat_df.apply(
#     lambda row: uniprot_ids.get((row['ECNumber'], row['Organism']), row['UniprotID']),
#     axis=1
# )

# # Retrieve sequences based on Uniprot IDs
# unique_uniprot_ids = kcat_df['UniprotID'].dropna().unique()
# for uniprot_id in unique_uniprot_ids:
#     sequence = get_uniprot_sequence(uniprot_id)
#     if sequence:
#         uniprot_sequences[uniprot_id] = sequence

# # Add sequences to the dataframe
# kcat_df['Sequence'] = kcat_df['UniprotID'].map(uniprot_sequences)

# # Save the updated dataframe back to a TSV file
# kcat_df.to_csv("../../Data/database/Kcat_combination_0731_st_p1.tsv", sep='\t', index=False)

# print("UniprotID and Sequence columns have been updated.")
# # 417min

Error retrieving Uniprot ID for EC 3.1.8.1 and organism Homo sapiens: HTTPSConnectionPool(host='rest.uniprot.org', port=443): Read timed out. (read timeout=None)
Error retrieving Uniprot ID for EC 2.7.7.60 and organism Plasmodium falciparum: HTTPSConnectionPool(host='rest.uniprot.org', port=443): Max retries exceeded with url: /uniprotkb/search?query=ec%3A2.7.7.60+AND+organism%3APlasmodium+falciparum+AND+reviewed%3Atrue&format=json&fields=accession (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:1000)')))
Error retrieving Uniprot ID for EC 1.1.1.184 and organism Thermotoga maritima: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error retrieving Uniprot ID for EC 1.14.14.82 and organism Arabidopsis thaliana: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error retrieving sequence for Uniprot ID Q45VU1 Q84F88: URL can't contain control characters. '/uni

In [4]:
# import pandas as pd

# # 读取包含 UniprotID 和 Sequence 列的文件
# kcat_df = pd.read_csv("../../Data/database/Kcat_combination_0731_st_p1.tsv", sep='\t')

# # 删除没有 UniprotID 和 Sequence 的条目
# filtered_kcat_df = kcat_df.dropna(subset=['UniprotID', 'Sequence'])

# # 保存过滤后的数据到新文件
# filtered_kcat_df.to_csv("../../Data/database/Kcat_combination_st_filtered.tsv", sep='\t', index=False)
# filtered_kcat_df

,ECNumber,EnzymeType,Organism,Smiles,Substrate,UniprotID,Value,Sequence
5,2.1.1.45,mutant H199A/N229D,Lactobacillus casei,C1C2CN(CN2C3=C(N1)N=C(NC3=O)N)C4=CC=C(C=C4)C(=...,"5,10-Methylenetetrahydrofolate",P00469,1.760000,MLEQPYLDLAKKVLDEGHFKPDRTHTGTYSIFGHQMRFDLSKGFPL...
9,4.2.1.51,"mutant S99T pheA, His-tagged",Corynebacterium glutamicum,C1=CC(C=CC1O)(CC(=O)C(=O)O)C(=O)O,Prephenate,P10341,0.001833,MSDAPTVVAYLGPAGTFTEEALYKFADAGVFGDGEIEQLPAKSPQE...
15,1.3.1.104,mutant W311A C-terminal His-tag,Homo sapiens,CCCCCC=CC(=O)SCCNC(=O)CCNC(=O)C(C(C)(C)COP(=O)...,trans-Oct-2-enoyl-CoA,Q9BV79,0.310000,MWVCSTLWRVRTPARQWRGLLPASGCHGPAASSYSASAEPARVRAL...
18,2.3.1.4,mutant V125R,Aspergillus fumigatus,C(C1C(C(C(C(O1)O)N)O)O)OP(=O)(O)O,alpha-D-glucosamine 6-phosphate,Q4WCU5,0.120000,MTNATIAPTTTAAPVTKSVDAPTADENTPLFSPSLISPDVLAVLPA...
22,1.1.1.18,mutant Y235F,Bacillus subtilis,C1=CC(=C[N+](=C1)C2C(C(C(O2)COP(=O)([O-])OP(=O...,NAD+,P26935,34.000000,MSLRIGVIGTGAIGKEHINRITNKLSGAEIVAVTDVNQEAAQKVVE...
...,...,...,...,...,...,...,...,...
38313,4.1.3.4,mutant R41Q,Homo sapiens,CC(C)(COP(=O)(O)OP(=O)(O)OCC1C(C(C(O1)N2C=NC3=...,3-Hydroxy-3-methylglutaryl-CoA,P35914,0.000400,MAAMRKALPRRLVGLASLRAVSTSSMGTLPKRVKIVEVGPRDGLQN...
38317,4.2.2.21,mutant H345A His8-tag,Bacteroides thetaiotaomicron,CC(=O)NC1C(C(C(OC1O)OS(=O)(=O)O)O)OC2C(C(C(C(O...,Chondroitin 4-sulfate,C5G6D7,63.800000,MLILSFLCPAFLNAQIVTDERMFSFEEPQLPACITGVQSQLGISGA...
38322,1.2.3.3,wildtype,Lactobacillus plantarum,CC(=O)C(=O)[O-],Pyruvate,P37063,17.900000,MVMKQTKQTNILAGAAVIKVLEAWGVDHLYGIPGGSINSIMDALSA...
38327,3.4.22.28,"wildtype 3Cpro (aa 1541-1723), C-terminal His-tag",Coxsackievirus B3,CC(C)CC(C(=O)NC(CCC(=O)N)C(=O)NC(CO)C(=O)NCC(=...,Dabcyl-KTSAVLQSGFRKME-Edans,P03313,0.006200,MGAQVSTQKTGAHETRLNASGNSIIHYTNINYYKDAASNSANRQDF...


### Plan2:seq优先级：brenda>uniprot

In [1]:
#!/usr/bin/python
# coding: utf-8

# Author: LE YUAN
# Date: 2020-08-08

# This python script is to obtain protein sequence for each Kcat entries

import os
import re
import json
import requests
import time
from urllib import request
from zeep import Client
import hashlib
from multiprocessing import Pool
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
# import string
# import hashlib
# from SOAPpy import WSDL
# from SOAPpy import SOAPProxy ## for usage without WSDL file
from tqdm import tqdm


def uniprot_sequence(id) :
    '''
    This function is to obtain the protein sequence according to the protein id from Uniprot API
    https://www.uniprot.org/uniprot/A0A1D8PIP5.fasta
    https://www.uniprot.org/help/api_idmapping
    '''
    url = "https://www.uniprot.org/uniprot/%s.fasta" % id
    IdSeq = dict()

    try :
        data = request.urlopen(url)
        respdata = data.read().decode("utf-8").strip()
        IdSeq[id] =  "".join(respdata.split("\n")[1:])
    except :
        print(id, "can not find from uniprot!")
        IdSeq[id] = None
    # print(IdSeq[id])
    return IdSeq[id]
    
def uniprotID_entry() :
    '''
    get sequence by uniprotID
    '''
    with open("../../Data/database/Kcat_combination_0731_st.tsv", "r", encoding='utf-8') as file :
        combination_lines = file.readlines()[1:]

    uniprotID_list = list()
    uniprotID_seq = dict()
    uniprotID_noseq = list()

    i=0
    for line in combination_lines :
        data = line.strip().split('\t')
        uniprotID = data[5]

        if uniprotID :
        #     seq = uniprot_sequence('P49384')
            if ' ' in uniprotID :
                # i += 1  # 561
                # print(i)
                # print(uniprotID.split(' '))
                uniprotID_list += uniprotID.split(' ')
            else :
                # print(uniprotID)
                uniprotID_list.append(uniprotID)


    uniprotID_unique = list(set(uniprotID_list))
    for uniprotID in tqdm(uniprotID_unique) :
        i += 1
        # print(i)
        sequence = uniprot_sequence(uniprotID)
        if sequence :
            uniprotID_seq[uniprotID] = sequence
        else :
            uniprotID_noseq.append(uniprotID)

    # 2289 total
    print(len(uniprotID_seq))  # 2267
    print(len(uniprotID_noseq))  # 22
    print(uniprotID_noseq)
    # ['P0A5R0', 'P0C5C1', 'P51698', 'P96807', 'Q01745', 'P00892', 'D0B556', 'V5MWQ6', 'Q02469', 'P96223', 'P0A4Z2', 
    # 'P0A4X4', 'P96420', 'Q47741', 'O05783', 'A3S939', 'P0A4X6', 'P56967', 'O60344', 'P04804', 'O52310']
    '''['A0A0D1LMH2', 'R4NNM3', 'O05783', 'P0A4X6', 'D4ZTT4', 'P0C5C1', 'G2H480', 'P0A4Z2', 'Q02469', 'O60344', 'P96807', 'A4VVM9', 'P00892', 'P00431', 'P0A4X4', 'P56967', 'P0A5R0', 'Q47741', 'O52310', 'P96420', 'P51698', 'P09148']'''
    # check one by one

    with open('../../Data/database/uniprotID_entry.json', 'w') as outfile :
        json.dump(uniprotID_seq, outfile, indent=4)

def uniprotID_noseq() :
    '''
    retreive the sequence by uniprotID that failed to get sequence from uniprot
    output: uniprotID_entry_all.json
    '''
    with open('../../Data/database/uniprotID_entry.json', 'r') as infile :
        uniprotID_seq = json.load(infile)
    # ['P0A4X4', 'Q02469', 'P96420', 'A4VVM9', 'P56967', 'O60344', 'P0A4X6', 'P0C5C1', 'P96807', 'P51698', 'A0A0D1LMH2', 'P0A4Z2', 'P0A5R0', 'O52310', 'G2H480', 'P00892', 'R4NNM3', 'O05783', 'D4ZTT4', 'Q47741']
    print(len(uniprotID_seq))
    uniprotID_noseq = {'P0A5R0':'P9WIL4', 'A0A0D1LMH2':'KIU47843', 'R4NNM3':'Q9WYR9', 'O05783':'P9WIQ2', 'P0A4X6':'P9WQ80', 'D4ZTT4':'WP_006619367', 'P0C5C1':'P0A5I7', 'G2H480':'EGY27206', 'P0A4Z2':'P0A4Z3', 'Q02469':'Q07WU7', 'O60344':'P0DPD6-2', 'P96807':'Q7U2S5', 'A4VVM9':'D5AI50', 'P00892':'P0DP90', 'P0A4X4':'P0A4X5', 'P56967':'P0DH76', 'Q47741':'F2MMN9', 'O52310':'P0CL72', 'P96420':'P9WQB2', 'P51698':'A0A1L5BTC1', 'P09148':'P09148'}
    # O60344有四条序列。

    for uniprotID, mappedID in uniprotID_noseq.items() :
        sequence = uniprot_sequence(mappedID)
        print(uniprotID)
        print(sequence)
        if sequence :
            uniprotID_seq[uniprotID] = sequence
        else :
            print('No sequence found!---------------------------')
    #未找到 手动赋值
    uniprotID_seq['A0A0D1LMH2'] = 'MPTCTLHPLPYQADPAAYFARIRQAPGAVLLDSARPGAVRGRFDLLSAWPLCTLTPDAEEDGQHYLQRLREQLAGLGRAHLPAGVELPFAGGLIGYLSYDFGRRLERLPAIACDDLGLGDASLGLYAWALISDHQLQRSQLVFHPALPATERERLVALFEHDAPTVAGAFRLAAPMRGDLAAEDYRQAFERVQQYIHAGDCYQINLTQRFRAPCQGDPWAAYQALRAACPTPFSGYQVLADGTALLSFSPERFIRVSQGEVETRPIKGTRARSSDPVQDAANAAELLASTKDRAENLMIVDLLRNDLGRSCATGSVQVPELFSLESYPNVHHMVSSVTGRLAPGKDALDLIAGSFPGGSITGAPKIRAMQIIDELEPSRRALYCGSLLYVDVRGEMDSSIAIRSLLVKDGQVCCWGGGAVVADSQWEAEYEESITKVRVLLQTLQGL'
    uniprotID_seq['D4ZTT4'] = 'MSEQKFGVIGLAVMGENLALNVERNGFPVAVYNRTSAKTDEFMQKRAPGKNVKPAYTLEEFVASLERPRRILVMVKAGKPVDAVINQLKPLLDHDDMIIDGGNSLYEDTERRTKELEATGLGFMGMGVSGGEEGALWGPSLMPGGTQNSYQALEPILTKIAAQVDDGPCVTYIGAGGAGHYVKMVHNGIEYGDMQLIAEAYDLLRNVIGLNEQQLYEVFAEWNTTDELNSFLIEITADIFKQKDPETGKPLVDLVLDSAGQKGTGRWTVVSALELGVSIPTITAAVNSRIISSYKDERVAASQELPGPSANFDGDVTSFINKVRDALYCSKICSYAQGMALMGKASQEFNYNLNLGEIARIWKGGCIIRAGFLNKIKQAYDQNPQLPNLLLAPEFKHTILDRQSAWREVLVTANTMGIPVPAFSASLDYFDSYRRVSLPQNLTQAQRDYFGAHTYERIDKPRGQFFHTEWANVD'
    uniprotID_seq['G2H480'] = 'MSSRSTRASKLVTSGQQRLGGDRPNLSARPVTRAISHPGPLTRRDFLRLAGGAALGLGIVSGLPTWAAWGGRAWAAAVNPASGGPLPLPGQSGLFGLLAPTGPFTLTAAPQPGLLPATGGPLLTYMAEHEGRVYHNPVLVLEKGQDFAVRLVNELGAIGPAPGGHGDAHGAGHGGHTPAASSGSNDAGPDTIIHWHGVDCPWRQAGHPMYAVGPGGHYDYAFPITNRAGTYWYHPHPHGDTARQAYLGLASFFIVRDDEERAFARELDLTLGVTDIPLVLQDKRIGPDGGLVYTPTQDELFMGYLGDRVLVNGAHLPTLSAATRVYRFRLLNGSTARIFNLSFVPSGGAGGSKGKATARPPVLPMTLIANDGGFLPSPRQVDGLFLAPGERAEVLLDLRGLDVGEVAWLRNLPFDPMHNEMDHAGGGTGEAGGTGHGGGHGDAGTHAVAKGTGGDHGSAVPHGMTRATMPRLTDAAEGSGGHGAGGGAHGETGGLAEGGGYPILRVSVDRAERYDRKVPERFSSSGDAVGALADLPAFTRTLRLEADGKRWTIAGETFAMDRFPITIPDRRRELWAIENAARSMPHPMHLHGYFFRVRERRGSPAQVRALAADGAGRLPTDLGLKDTVLAWPGETVLADVDFGAPAYPGEQVFLFHCHNLEHEDQGMMVNVSLP'
    print(len(uniprotID_seq))  # 2289

    with open('../../Data/database/uniprotID_entry_all.json', 'w') as outfile :
        json.dump(uniprotID_seq, outfile, indent=4)

import re

def escape_special_characters(s):
    s = re.sub(r'\(.*?\)', '', s)
    s = re.sub(r'([+\-&|!(){}\[\]^"~*:?])', r'\\\1', s)
    return s



def seq_by_ec_organism(ec_number, organism, retries=3):
    base_url = "https://rest.uniprot.org/uniprotkb/search?"
    organism = escape_special_characters(organism)
    query = f"(ec:{ec_number} AND organism_name:{organism})"
    params = {
        "query": query,
        "format": "fasta",
        "size": 1
    }

    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            fasta_data = response.text
            # print(fasta_data)
            if fasta_data:
                fasta_lines = fasta_data.strip().split("\n")
                sequence_id = fasta_lines[0].lstrip(">").split("|")[1]
                sequence = "".join(fasta_lines[1:])
                # 返回包含Uniprot ID和序列的字典
                return {
                    'uniprot_id': sequence_id,
                    'sequence': sequence
                }
                break

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred: {http_err}")
        except requests.exceptions.ConnectionError as conn_err:
            print(f"Connection error occurred: {conn_err}")
        except requests.exceptions.Timeout as timeout_err:
            print(f"Timeout error occurred: {timeout_err}")
        except requests.exceptions.RequestException as req_err:
            print(f"An error occurred: {req_err}")
        except ValueError as val_err:
            print(val_err)

    return None

# def seq_by_brenda(ec, organism, retries=3):
#     email = '1055285901@qq.com'
#     password = 'LBXSQJLRTZ1124'
#     hashed_password = hashlib.sha256(password.encode("utf-8")).hexdigest()
#     wsdl = "https://www.brenda-enzymes.org/soap/brenda_zeep.wsdl"
#     client = Client(wsdl)

#     session = requests.Session()
#     retry_strategy = Retry(total=5, backoff_factor=1, status_forcelist=[502, 503, 504])
#     session.mount('https://', HTTPAdapter(max_retries=retry_strategy))

#     parameters = (email, hashed_password, "ecNumber*%s" % ec, "organism*%s" % organism, "sequence*", "noOfAminoAcids*", "firstAccessionCode*", "source*Swiss-Prot", "id*")
    
#     attempt = 0
#     while attempt < retries:
#         try:
#             entries = client.service.getSequence(*parameters)
#             sequences = []
#             if entries:
#                 for entry in entries:
#                     sequences.append(entry['sequence'])
#                 return sequences
#         except Exception as e:
#             print(f"Attempt {attempt + 1} failed: {e}")
#             attempt += 1
#             time.sleep(2 ** attempt)
#     return []

def process_entry(entry):
    ec, organism = entry
    result = seq_by_ec_organism(ec, organism)
    if result is not None:
        return (ec, organism), (result['uniprot_id'], result['sequence'])
    else:
        return (ec, organism), ('', '')

def nouniprotID_entry_uniprot() :
    # ec = '1.1.1.206'
    # organism = 'Datura stramonium'
    # seq_by_ec_organism(ec, organism)

    with open("../../Data/database/Kcat_combination_0731_st.tsv", "r", encoding='utf-8') as file:
        combination_lines = file.readlines()[1:]

    entries = list()
    for line in combination_lines:
        data = line.strip().split('\t')
        ec = data[0]
        organism = data[2]
        uniprotID = data[5]

        if not uniprotID:
            entries.append((ec, organism))

    entries_unique = set(entries)
    
    # 使用 Pool 来并行处理
    num_cores = min(16, len(entries_unique))  # 使用的核心数最多16
    print('cores:', num_cores)
    with Pool(num_cores) as pool:
        result = list(tqdm(pool.imap(process_entry, entries_unique), total=len(entries_unique)))
    IdSeq = dict()
    for item in result:
        (ec, organism), (uniprot_id, sequence) = item
        if uniprot_id:
            IdSeq[f"{ec}&{organism}"] = [uniprot_id, f'{sequence}']
        else:
            IdSeq[f"{ec}&{organism}"] = None
    with open('../../Data/database/nouniprotID_entry_all.json', 'w') as outfile:
        json.dump(IdSeq, outfile, indent=4)

# def nouniprotID_entry_brenda():
#     with open("../../Data/database/Kcat_combination_0731_st.tsv", "r", encoding='utf-8') as file:
#         combination_lines = file.readlines()[1:]

#     entries = []
#     for line in combination_lines:
#         data = line.strip().split('\t')
#         ec = data[0]
#         organism = data[2]
#         uniprotID = data[5]
#         if not uniprotID:
#             entries.append((ec, organism))

#     entries_unique = set(entries)
#     print(f"Total entries to process: {len(entries)}")
#     print(f"Unique entries to process: {len(entries_unique)}")

#     # Use Pool for parallel processing
#     num_cores = min(16, len(entries_unique))  # Limit cores to 16 or less
#     with Pool(num_cores) as pool:
#         result = list(tqdm(pool.imap(process_brenda_entry, entries_unique), total=len(entries_unique)))

#     # Convert results to dictionary
#     IdSeq = dict(result)

#     with open('../../Data/database/nouniprotID_entry_brenda.json', 'w') as outfile:
#         json.dump(IdSeq, outfile, indent=4)

# def process_brenda_entry(entry):
#     ec, organism = entry
#     sequence = seq_by_brenda(ec, organism)
#     return ec + '&' + organism, sequence

# def combine_sequence1() :
#     with open('../../Data/database/uniprotID_entry_all.json', 'r') as file1:
#         uniprot_file1 = json.load(file1)

#     with open('../../Data/database/nouniprotID_entry_all.json', 'r') as file2:  # By Uniprot API
#         nouniprot_file2 = json.load(file2)

#     with open("../../Data/database/Kcat_combination_0731_st.tsv", "r", encoding='utf-8') as file4 :
#         Kcat_lines = file4.readlines()[1:]
#     i = 0
#     j = 0
#     n = 0
#     entries = list()
#     for line in Kcat_lines :
#         data = line.strip().split('\t')
#         ECNumber, EnzymeType, Organism, Smiles = data[0], data[1], data[2], data[3]
#         Substrate, UniprotID, Value = data[4], data[5], data[6]

#         RetrievedSeq = ''
#         entry = dict()
#         # print(UniprotID)
#         if UniprotID :
#             try :  # because a few (maybe four) UniprotIDs have no ID as the key 
#                 if ' ' not in UniprotID :
#                     RetrievedSeq = uniprot_file1[UniprotID]
#                 else :
#                     # print(UniprotID)
#                     RetrievedSeq1 = uniprot_file1[UniprotID.split(' ')[0]]
#                     RetrievedSeq2 = uniprot_file1[UniprotID.split(' ')[1]]
#                     if RetrievedSeq1 == RetrievedSeq2 :
#                         RetrievedSeq = RetrievedSeq1
#             except :
#                 continue

#         else :
#             if nouniprot_file2[ECNumber+'&'+Organism] :
#                 # print(nouniprot_file2[ECNumber+'&'+Organism])
#                 UniprotID, RetrievedSeq = nouniprot_file2[ECNumber+'&'+Organism]
#             else:
#                 RetrievedSeq = ''

#         try:  # local variable 'RetrievedSeq' referenced before assignment
#             if ' ' in UniprotID:
#                 UniprotID = UniprotID.split(' ')[0]
#             if EnzymeType == 'wildtype' and RetrievedSeq != '':  # 21108 for all, 9529 wildtype, 11579 mutant (EnzymeType != 'wildtype')
#                 i += 1
#                 entry = {
#                     'ECNumber': ECNumber,
#                     'Organism': Organism,
#                     'Smiles': Smiles,
#                     'Substrate': Substrate,
#                     'Sequence': RetrievedSeq,
#                     'Type': 'wildtype',
#                     'UniprotID': UniprotID,
#                     'Value': Value,
#                 }

#                 entries.append(entry)

#             if EnzymeType != 'wildtype' and RetrievedSeq != '':
#                 sequence = RetrievedSeq

#                 mutantSites = EnzymeType.split('/')
#                 # print(mutantSites)

#                 mutant1_1 = [mutantSite[1:-1] for mutantSite in mutantSites]
#                 mutant1_2 = [mutantSite for mutantSite in mutantSites]
#                 mutant1 = [mutant1_1, mutant1_2]
#                 mutant2 = set(mutant1[0])
#                 if len(mutant1[0]) != len(mutant2) :
#                     print(mutant1)
#                     n += 1
#                     print(str(n) + '---------------------------')  # some are mapped, some are not mapped. R234G/R234K (60, 43 mapped, 17 not mapped)

#                 mutatedSeq = sequence
#                 for mutantSite in mutantSites :
#                     if mutatedSeq[int(mutantSite[1:-1])-1] == mutantSite[0] :
#                         # pass
#                         mutatedSeq = list(mutatedSeq)
#                         mutatedSeq[int(mutantSite[1:-1])-1] = mutantSite[-1]
#                         mutatedSeq = ''.join(mutatedSeq)
#                         if not mutatedSeq :
#                             print('-------------')
#                     else :
#                         mutatedSeq = ''

#                 if mutatedSeq :   
#                     entry = {
#                         'ECNumber': ECNumber,
#                         'Organism': Organism,
#                         'Smiles': Smiles,
#                         'Substrate': Substrate,
#                         'Sequence': mutatedSeq,
#                         'Type': 'mutant',
#                         'UniprotID': UniprotID,
#                         'Value': Value,
#                     }
#                     entries.append(entry)

#         except:
#             continue

#     print('mutant',i)

#     print('total', len(entries))   # 17010  including 9529 wildtype and 7481 mutant

#     with open('../../Data/database/Kcat_combination_0918_wildtype_mutant_0826.json', 'w') as outfile :
#         json.dump(entries, outfile, indent=4)
def combine_sequence() :
    with open('../../Data/database/uniprotID_entry_all.json', 'r') as file1:
        uniprot_file1 = json.load(file1)

    with open('../../Data/database/nouniprotID_entry_all.json', 'r') as file2:  # By Uniprot API
        nouniprot_file2 = json.load(file2)

    with open("../../Data/database/Kcat_combination_0731_st.tsv", "r", encoding='utf-8') as file4 :
        Kcat_lines = file4.readlines()[1:]

    i = 0
    j = 0
    n = 0
    entries = list()
    for line in Kcat_lines :
        data = line.strip().split('\t')
        ECNumber, EnzymeType, Organism, Smiles = data[0], data[1], data[2], data[3]
        Substrate, UniprotID, Value = data[4], data[5], data[6]

        RetrievedSeq = ''
        entry = dict()
        # print(UniprotID)
        if UniprotID :
            # print(UniprotID)
            try :  # because a few (maybe four) UniprotIDs have no ID as the key 
                if ' ' not in UniprotID :
                    RetrievedSeq = [uniprot_file1[UniprotID]]
                    # print(RetrievedSeq)
                else :
                    # print(UniprotID)
                    RetrievedSeq1 = [uniprot_file1[UniprotID.split(' ')[0]]]
                    RetrievedSeq2 = [uniprot_file1[UniprotID.split(' ')[1]]]
                    if RetrievedSeq1 == RetrievedSeq2 :
                        RetrievedSeq = RetrievedSeq1
                    # if len(RetrievedSeq) == 1:
                    #     print(RetrievedSeq)
            except :
                continue

        else :
            if nouniprot_file2[ECNumber+'&'+Organism] :
                # print(nouniprot_file2[ECNumber+'&'+Organism])
                if len(nouniprot_file2[ECNumber+'&'+Organism]) == 1 :
                    RetrievedSeq = nouniprot_file2[ECNumber+'&'+Organism]
                    # print(RetrievedSeq)
                else :
                    RetrievedSeq = ''

        # print(RetrievedSeq)
        try:  # local variable 'RetrievedSeq' referenced before assignment
            if len(RetrievedSeq) == 1 and EnzymeType == 'wildtype':  # 21108 for all, 9529 wildtype, 11579 mutant (EnzymeType != 'wildtype')
                sequence = RetrievedSeq
                i += 1
                entry = {
                    'ECNumber': ECNumber,
                    'Organism': Organism,
                    'Smiles': Smiles,
                    'Substrate': Substrate,
                    'Sequence': sequence[0],
                    'Type': 'wildtype',
                    'Value': Value,
                    # 'Unit': Unit,
                }

                entries.append(entry)

            if len(RetrievedSeq) == 1 and EnzymeType != 'wildtype':
                sequence = RetrievedSeq[0]

                mutantSites = EnzymeType.split('/')
                # print(mutantSites)

                mutant1_1 = [mutantSite[1:-1] for mutantSite in mutantSites]
                mutant1_2 = [mutantSite for mutantSite in mutantSites]
                mutant1 = [mutant1_1, mutant1_2]
                mutant2 = set(mutant1[0])
                if len(mutant1[0]) != len(mutant2) :
                    print(mutant1)
                    n += 1
                    print(str(n) + '---------------------------')  # some are mapped, some are not mapped. R234G/R234K (60, 43 mapped, 17 not mapped)

                mutatedSeq = sequence
                for mutantSite in mutantSites :
                    # print(mutantSite)
                    # print(mutatedSeq[int(mutantSite[1:-1])-1])
                    # print(mutantSite[0])
                    # print(mutantSite[-1])
                    if mutatedSeq[int(mutantSite[1:-1])-1] == mutantSite[0] :
                        # pass
                        mutatedSeq = list(mutatedSeq)
                        mutatedSeq[int(mutantSite[1:-1])-1] = mutantSite[-1]
                        mutatedSeq = ''.join(mutatedSeq)
                        if not mutatedSeq :
                            print('-------------')
                    else :
                        mutatedSeq = ''

                if mutatedSeq :     
                    entry = {
                        'ECNumber': ECNumber,
                        'Organism': Organism,
                        'Smiles': Smiles,
                        'Substrate': Substrate,
                        'Sequence': mutatedSeq,
                        'Type': 'mutant',
                        'Value': Value,
                        # 'Unit': Unit,
                    }

                    entries.append(entry)

        except:
            continue
    print(i)

    print(len(entries))   # 17010  including 9529 wildtype and 7481 mutant

    with open('../../Data/database/Kcat_combination_0918_wildtype_mutant_0826.json', 'w') as outfile :
        json.dump(entries, outfile, indent=4)

def check_substrate_seq() :
    with open('../../Data/database/Kcat_combination_0918_wildtype_mutant_0826.json', 'r') as file :
        datasets = json.load(file)

    substrate = [data['Substrate'].lower() for data in datasets]
    sequence = [data['Sequence'] for data in datasets]
    organism = [data['Organism'].lower() for data in datasets]
    EC_number = [data['ECNumber'] for data in datasets]

    unique_substrate = len(set(substrate))
    unique_sequence = len(set(sequence))
    unique_organism = len(set(organism))
    unique_EC_number = len(set(EC_number))

    print('The number of unique substrate:', unique_substrate)
    print('The number of unique sequence:', unique_sequence)
    print('The number of unique organism:', unique_organism)
    print('The number of unique EC Number:', unique_EC_number)
    
    with open('seq.txt', 'w') as f:
        f.write('\n'.join(set(sequence)))  # 使用换行符将列表中的每个元素分隔开
    # The number of unique substrate: 2706
    # The number of unique sequence: 7857
    # The number of unique organism: 856
    # The number of unique EC Number: 1706
    # @240820:
    # 13428
    # 21208
    # The number of unique substrate: 3512
    # The number of unique sequence: 10172
    # The number of unique organism: 1041
    # The number of unique EC Number: 2126
    # @240826：
    # mutant 18227
    # total 26927
    # The number of unique substrate: 3892
    # The number of unique sequence: 12979
    # The number of unique organism: 1740
    # The number of unique EC Number: 2273
if __name__ == "__main__" :
    # uniprotID_entry()#file1 #2330 in total, 21(includeQ9ZAG3 手动添加) not found - @240820
    # uniprotID_noseq()#file1 共20个 已完成
    # nouniprotID_entry_uniprot()#file2 #
    # nouniprotID_entry_brenda()#file3 最后没用上
    combine_sequence()
    check_substrate_seq()

[['49', '49'], ['R49C', 'R49L']]
1---------------------------
[['292', '292'], ['E292Q', 'E292A']]
2---------------------------
[['87', '87'], ['E87Q', 'E87G']]
3---------------------------
[['172', '172'], ['A172E', 'A172N']]
4---------------------------
[['258', '258'], ['R258A', 'R258H']]
5---------------------------
[['258', '258', '258'], ['R258A', 'R258L', 'R258K']]
6---------------------------
[['258', '258'], ['R258A', 'R258F']]
7---------------------------
[['258', '258'], ['R258L', 'R258F']]
8---------------------------
[['190', '190'], ['A190T', 'A190S']]
9---------------------------
[['83', '83'], ['Y83F', 'Y83L']]
10---------------------------
4779
7702
The number of unique substrate: 1510
The number of unique sequence: 3862
The number of unique organism: 485
The number of unique EC Number: 937


In [8]:
import pandas as pd
df = pd.read_csv('../../Data/database/Kcat_combination_0731_st.tsv',sep = '\t')
# df1 = pd.read_csv('../../Data/database/Kcat_sabio_clean_unisubstrate.tsv',sep = '\t')
# df2 = pd.read_csv('../../Data/database/Kcat_brenda_clean.tsv',sep = '\t')
# for column in df.columns:
#     print(f"统计列：{column}")
#     print(df[column].value_counts())
unique_counts = df.nunique()
print("sabio每列的唯一值数量:")
print(unique_counts)
# unique_counts = df2.nunique()
# print("brenda每列的唯一值数量:")
# print(unique_counts)


sabio每列的唯一值数量:
ECNumber       2883
EnzymeType    15798
Organism       2586
Smiles         4691
Substrate      4691
UniprotID      2245
Value          8374
Unit              1
dtype: int64


In [6]:
import json

# 载入JSON文件
with open('../../Data/database/nouniprotID_entry_all.json', 'r') as file:
    data = json.load(file)

# 统计值为null的条目数量
null_count = sum(1 for key in data if data[key] is None)

# 显示结果
print(f"值为null的条目数量: {null_count}")
print(f"总条目数量: {len(data)}")



值为null的条目数量: 2510
总条目数量: 8283


In [5]:
username = '1055285901@qq.com'
password = 'LBXSQJLRTZ1124'

from zeep import Client

# 使用HTTPS地址
url = 'https://www.brenda-enzymes.org/soap/brenda_server.php?wsdl'
client = Client(url)
parameter = "ecNumber*1.1.1.1#organism*Homo sapiens"

# 根据具体的API方法调用调整
try:
    response = client.service.getKmValue(parameter, username, password)
    print(response)
except Exception as e:
    print(f"Error: {e}")


Error: Service has no operation 'getKmValue'


/home/wuke/anaconda3/envs/protssn/lib/python3.12/site-packages/zeep/wsdl/wsdl.py:352: UserWarning: The wsdl:message for '{ws.brenda}getReferenceByIdRequest' contains an invalid part ('id'): invalid xsd type or elements
  warnings.warn(str(exc))
/home/wuke/anaconda3/envs/protssn/lib/python3.12/site-packages/zeep/wsdl/wsdl.py:352: UserWarning: The wsdl:message for '{ws.brenda}getReferenceByIdResponse' contains an invalid part ('return'): invalid xsd type or elements
  warnings.warn(str(exc))
/home/wuke/anaconda3/envs/protssn/lib/python3.12/site-packages/zeep/wsdl/wsdl.py:352: UserWarning: The wsdl:message for '{ws.brenda}getReferenceByPubmedIdRequest' contains an invalid part ('id'): invalid xsd type or elements
  warnings.warn(str(exc))
/home/wuke/anaconda3/envs/protssn/lib/python3.12/site-packages/zeep/wsdl/wsdl.py:352: UserWarning: The wsdl:message for '{ws.brenda}getReferenceByPubmedIdResponse' contains an invalid part ('return'): invalid xsd type or elements
  warnings.warn(str(exc)

In [7]:
from zeep import Client

# 使用BRENDA的WSDL URL
client = Client('https://www.brenda-enzymes.org/soap/brenda_server.php?wsdl')

# 打印所有服务和操作
for service in client.wsdl.services.values():
    print(f"Service: {service.name}")
    for port in service.ports.values():
        operations = sorted(
            port.binding._operations.keys(),
            key=lambda x: x.lower()
        )
        for operation in operations:
            print(f" - {operation}")


Service: brenda_webservice
